# CI integration test: regression

Mirrors `examples/regression_example.ipynb`'s pipeline (synthetic
amplitude-varying ERP -> tfrecords with `target_type='float'` -> LFCNN
cross-validation -> prediction -> pattern interpretation), always on a
lightweight synthetic dataset generated directly in sensor space (no MNE
forward solution / download, unlike the tutorial notebook's source-space
simulation). Not a tutorial -- see the notebook in `examples/` for that.
Independent of the other CI notebooks (uses its own `data_id`).

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import mne
mne.set_log_level(verbose='CRITICAL')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

import mneflow
print(mneflow.__version__)

In [ ]:
n_epochs = int(os.environ.get('MNEFLOW_N_EPOCHS', '3'))
path = os.environ.get('MNEFLOW_DATA_PATH', '/tmp/mneflow_ci/')
data_id = 'sim_regression_example'

In [ ]:
# Lightweight synthetic stand-in for the source-simulated dataset in
# examples/regression_example.ipynb: an amplitude-jittered ERP waveform on
# simulated grad channels. `y` (the per-trial amplitude) is the regression
# target.
sfreq = 600.0
n_ch = 204
ch_names = [f'MEG{i:04d}' for i in range(n_ch)]
info = mne.create_info(ch_names, sfreq, ch_types='grad')

n_events = 200
times = np.arange(120) / sfreq - 0.05

def data_fun(t, dip_amp, freq, peak_ind=.0525, peak_hw=0.00065):
    sin = dip_amp * 1e-9 * np.sin(2 * np.pi * freq * t)
    exp = np.exp(-(t - peak_ind) ** 2 / peak_hw)
    return sin * exp

rng = np.random.RandomState(0)
source_time_series = data_fun(times, 5, 19)
amps = np.round(0.50 * rng.rand(n_events) + 1. - 0.25, 2)
topo = rng.randn(n_ch, 1) * 1e-12
waveforms = amps[:, None] * source_time_series
X = topo[None, :, :] * waveforms[:, None, :] + rng.randn(n_events, n_ch, len(times)) * 5e-13
y = amps
print("X:", X.shape, " y:", y.shape)

In [ ]:
import_opt = dict(path=path,
                  data_id=data_id,
                  fs=sfreq,
                  input_type='trials',
                  target_type='float',
                  n_folds=5,
                  scale=True,
                  crop_baseline=False,
                  scale_interval=(0, 60),
                  overwrite=True,
                  test_set='holdout')

meta = mneflow.produce_tfrecords((X, y), **import_opt)

In [ ]:
dataset = mneflow.Dataset(meta, train_batch=50)

lf_params = dict(n_latent=32,
                  filter_length=128,
                  nonlin=tf.nn.relu,
                  padding='SAME',
                  pooling=32,
                  stride=24,
                  pool_type='avg',
                  model_path=import_opt['path'],
                  dropout=.25,
                  l2_scope=['tconv', 'dmx'],
                  l2_lambda=3e-3,
                  l1_scope=['fc'],
                  l1_lambda=3e-3)

meta.update(model_specs=lf_params)

model = mneflow.LFCNN(meta, dataset)
model.build(optimizer='adam', learn_rate=3e-4)

In [ ]:
model.train(n_epochs=n_epochs, eval_step=50, early_stopping=5, mode='cv', collect_patterns=True)

In [ ]:
test_loss, test_metric = model.evaluate(meta.data['test_paths'])
print("Test set: Loss = {:.4f}  Metric = {:.4f}".format(test_loss, test_metric))

In [ ]:
import matplotlib.pyplot as plt

y_true, y_pred = model.predict()
f, ax = plt.subplots(1, 2)
ax[0].scatter(y_true, y_pred)
ax[1].scatter(y_true, y_pred - y_true)
plt.show()

In [ ]:
model.meta.plot_spatial_patterns('weight', sensor_layout='Vectorview-grad')

In [ ]:
model.meta.plot_spectra(method='weight', log=False, freqs_lim=(1, 45))

In [ ]:
model.meta.plot_timecourses(freqs_lim=(1, 45), method='weight', average_over=None)

In [ ]:
model.meta.explore_components(sorting='weight', sensor_layout='Vectorview-grad', diff=False, n_cols=1)